In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

bronze_df = spark.read.option("multiline","true").json("Files/Covid_bronze.json")

display(bronze_df)

StatementMeta(, ecb3b73d-5ef5-406c-94da-216b968ff199, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2622c9c0-f226-4866-b645-da4c1e9b7120)

In [2]:
# Flatten nested fields from 'CountryInfo'

silver_df = (
    bronze_df
    .withColumn("country_id" , F.col("countryInfo._id"))
    .withColumn("iso2" , F.col("countryInfo.iso2"))
    .withColumn("iso3" , F.col("countryInfo.iso3"))
    .withColumn("latitude" , F.col("countryInfo.lat"))
    .withColumn("longitude" , F.col("countryInfo.long"))
    .withColumn("flag_url" ,  F.col("countryInfo.flag"))
    .drop("countryInfo")
)

display(silver_df)

StatementMeta(, ecb3b73d-5ef5-406c-94da-216b968ff199, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 96b57ff9-b72e-4b3c-bfeb-845f87e4b4bf)

In [4]:
silver_df = (
    silver_df
    .withColumn("death_rate_pct", F.round((F.col("deaths") / F.col("cases")) * 100 , 2))
    .withColumn("recovery_rate_pct" , F.round((F.col("recovered") / F.col("cases")) * 100, 2))
    .withColumn("active_rate_pct" , F.round((F.col("active") / F.col("cases")) * 100 , 2 ))
    .withColumn("test_per_thousand", F.round((F.col("tests") / F.col("population")) * 100 , 2 ))
    .withColumn("cases_per_million", F.round(F.col("casesPerOneMillion"), 2 ))
)

silver_df = silver_df.na.drop(subset=["cases","deaths","recovered","population"])

display(silver_df)

StatementMeta(, ecb3b73d-5ef5-406c-94da-216b968ff199, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9a032bf7-d095-4b86-ad5d-167d4813f53f)

In [5]:
silver_table = "Covid_Silver"
silver_df.write.mode("overwrite").saveAsTable(silver_table)

print(f"Silver table {silver_table} created successfully")

StatementMeta(, ecb3b73d-5ef5-406c-94da-216b968ff199, 7, Finished, Available, Finished)

Silver table Covid_Silver created successfully
